# Brainrot-V — generate

Produces evaluation videos with the same run record Modal did, so results stay
citeable. Scoring and plots are in `colab_evaluate.ipynb`.

**Run the cells in order.** Cell 4 is the only one that costs GPU time.

### Built to be interrupted

Colab Pro has no background execution, so the session ends when the browser
does. Everything assumes that:

- **Checkpointed per video.** A finished video is skipped next time. A crash
  costs only the run that was in flight, 20–24 minutes at worst.
- **Atomic writes.** A runtime that dies mid-write leaves the previous file
  intact, never a half-written one.
- **Verified, not trusted.** A run counts as finished only if its manifest says
  so *and* the video still hashes to what the manifest recorded.
- **`STATUS.md` on Drive** is rewritten after every run, so you can see progress
  from the Drive web UI without opening the notebook.

To resume after any crash: re-run cells 1–4. Nothing else.

### Three run sets

| `RUN_SET` | Runs | GPU time | Answers |
| --- | ---: | ---: | --- |
| `bridge` | 2 | ~0.8 h | Does Colab reproduce Modal's numbers? **Run first** |
| `pilot` | 8 | ~3.2 h | Does Brainrot-V beat the rolling baseline? |
| `final` | 32 | ~10.8 h | The real comparison at 25 s |

An A100 costs roughly **14 compute units per hour** and Colab Pro gives 100 a
month, so: bridge ≈ 11 units, pilot ≈ 45, final ≈ 151.

### Why `bridge` first

Every existing result came from an A100-**80**GB with flash-attn. Colab gives an
A100-**40**GB, which takes a different branch at `brainrot_inference.py:205`
(the text encoder is swapped in and out rather than kept resident), and
flash-attn may not compile, in which case attention falls back to another
kernel. Neither is a bug, but it means Colab videos are not automatically
comparable with the clips already in `output/llm-methods-30s/`.

`bridge` regenerates `m0` at seeds 1 and 4 so you can check against the Modal
flicker values — **3.21** and **0.86**. Identical runs already differ by
34.7–40.2 dB, so the question is whether the gap sits inside that noise.

### Put on Drive first, at `MyDrive/brainrot-v/00_setup/`

- `code/causal_forcing.zip`
  ```
  cd D:/thesis/src && zip -r causal_forcing.zip causal_forcing \
    -x '*/.modal-client/*' '*/output/*' '*/__pycache__/*' '*.mp4' '*.pt' '*.pth'
  ```
- `models/` — 23 GB. Cell 2 downloads them if absent, which needs Google One
  100 GB; Drive's free 15 GB will not fit them.

## 1 · Settings

In [ ]:
RUN_SET = "bridge"        # "bridge" | "pilot" | "final"
ROOT    = "/content/drive/MyDrive/brainrot-v"
REPO    = "/content/causal_forcing"

## 2 · Setup — Drive, GPU, code, packages, models

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

# ---- code (holds colab_lib, so it is unzipped before anything imports it) ---
archive = Path(ROOT) / "00_setup" / "code" / "causal_forcing.zip"
if not archive.exists():
    raise FileNotFoundError(f"Upload causal_forcing.zip to {archive.parent}")
if Path(REPO).exists():
    shutil.rmtree(REPO)
with zipfile.ZipFile(archive) as zf:
    zf.extractall("/content")
repo = Path(REPO)
os.chdir(repo)

sys.path.insert(0, str(repo / "notebooks"))
import colab_lib as cl
ws = cl.Workspace(ROOT).ensure()
print("workspace  ", ws.root)

# ---- GPU -------------------------------------------------------------------
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                       "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.strip()
gpu_name, gpu_mib = [x.strip() for x in info.split(",")]
gpu_gb = int(gpu_mib) / 1024
if "A100" not in gpu_name.upper():
    raise RuntimeError(f"Got {gpu_name}, not an A100. Generation would take hours "
                       "per video and waste units. Recycle the runtime and retry.")

# Which branch brainrot_inference.py:205 takes on this card.
MEMORY_PATH = "resident" if gpu_gb > 60 else "dynamic_swap"
print(f"GPU        {gpu_name} ({gpu_gb:.0f} GB), memory path {MEMORY_PATH}")

PROMPT_30S = "prompts/brainrot_v_long_horizon_60s.txt"
prompt_sha = hashlib.sha256((repo / PROMPT_30S).read_bytes()).hexdigest()
assert prompt_sha.startswith("e4b595af"), (
    f"Prompt differs from the existing m0 runs (got {prompt_sha[:16]}); the "
    "bridge check would compare different videos")
print(f"prompt     {prompt_sha[:12]} matches the eviction-screen runs")

# ---- packages --------------------------------------------------------------
print("packages   installing (slow on a cold session) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=True)

# flash-attn has no prebuilt wheel for this combination and takes 30-60 min to
# compile, so it is built once and kept on Drive for later sessions.
wheel = next(iter(sorted(ws.wheels.glob("flash_attn-2.7.4*.whl"))), None)
if wheel:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(wheel)],
                   check=False)
else:
    print("           building flash-attn once (30-60 min) ...")
    build = Path("/content/wheels")
    if subprocess.run([sys.executable, "-m", "pip", "wheel",
                       "flash-attn==2.7.4.post1", "--no-build-isolation",
                       "-w", str(build)], check=False).returncode == 0:
        for built in build.glob("flash_attn-*.whl"):
            shutil.copy2(built, ws.wheels / built.name)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            str(built)], check=False)

try:
    import flash_attn  # noqa: F401
    FLASH_ATTN = "2.7.4.post1"
except ModuleNotFoundError:
    FLASH_ATTN = None

import torch
print(f"torch      {torch.__version__}")
print(f"flash-attn {FLASH_ATTN or 'ABSENT -> scaled_dot_product_attention'}")

# ---- models ----------------------------------------------------------------
wan = ws.models / "wan_models" / "Wan2.1-T2V-1.3B"
ckpt = ws.models / "checkpoints" / "framewise" / "ar_diffusion.pt"
SIZES = {
    wan / "models_t5_umt5-xxl-enc-bf16.pth": 11_361_920_000,
    wan / "diffusion_pytorch_model.safetensors": 5_680_000_000,
    wan / "Wan2.1_VAE.pth": 510_000_000,
    ckpt: 5_680_000_000,
}
ok = lambda p, n: p.exists() and abs(p.stat().st_size - n) <= n * 0.05

if not all(ok(p, n) for p, n in SIZES.items()):
    print("models     downloading ~23 GB to Drive (one time) ...")
    from huggingface_hub import hf_hub_download, snapshot_download
    if not all(ok(p, n) for p, n in SIZES.items() if p != ckpt):
        snapshot_download(repo_id="Wan-AI/Wan2.1-T2V-1.3B", local_dir=str(wan))
    if not ok(ckpt, SIZES[ckpt]):
        hf_hub_download(repo_id="zhuhz22/Causal-Forcing",
                        filename="framewise/ar_diffusion.pt",
                        local_dir=str(ws.models / "checkpoints"))
for p, n in SIZES.items():
    assert ok(p, n), f"{p.name} missing or truncated"

# Copy to session disk: each run is a fresh subprocess that reloads the weights,
# and repeated 23 GB reads over Drive cost more in billed GPU time than this.
LOCAL = Path("/content/models")
for src in SIZES:
    dst = LOCAL / src.relative_to(ws.models)
    if not ok(dst, SIZES[src]):
        dst.parent.mkdir(parents=True, exist_ok=True)
        print(f"           staging {src.name} ({src.stat().st_size/2**30:.1f} GiB)")
        shutil.copy2(src, dst)
CHECKPOINT = LOCAL / ckpt.relative_to(ws.models)

link = repo / "wan_models"              # brainrot_inference.py resolves Wan here
if link.is_symlink():
    link.unlink()
elif link.exists():
    shutil.rmtree(link)
link.symlink_to(LOCAL / "wan_models", target_is_directory=True)
print("models     ready")

## 3 · The plan

Shows what will run and what already finished. A run counts as finished only if
its manifest says `succeeded` **and** the video on Drive still hashes to what the
manifest recorded — on Modal a stale manifest was twice mistaken for a live
result.

In [ ]:
CONFIG = "configs/llm_methods/brainrot_v_m0-baseline.yaml"


def job(run_id, mode, seed, frames, prompt=PROMPT_30S):
    return dict(run_id=run_id, mode=mode, seed=seed, latent_frames=frames,
                sampling_steps=50, config_path=CONFIG, prompt_file=prompt)


RUN_SETS = {
    # Modal m0 flicker for comparison: seed 1 = 3.21, seed 4 = 0.86
    "bridge": [job(f"bridge-m0-s{s}-30s", "brainrot_v", s, 121) for s in (1, 4)],

    # The baseline arm, pairing against the eight m0 clips that already exist
    "pilot": [job(f"pilot-baseline-s{s}-30s", "baseline", s, 121) for s in range(8)],

    # 4 prompts x 4 seeds x 2 arms at 25 s. The prompts are not written yet, so
    # this set reports them missing until they are.
    "final": [job(f"final-{arm}-p{p}-s{s}-25s", mode, s, 101,
                  f"prompts/eval25/prompt{p}.txt")
              for p in (1, 2, 3, 4) for s in range(4)
              for arm, mode in (("baseline", "baseline"),
                                ("brainrotv", "brainrot_v"))],
}
JOBS = RUN_SETS[RUN_SET]
HOURS_PER = {121: 0.403, 101: 0.337}        # measured 12.0 s per latent frame

print(f"set: {RUN_SET}\n")
missing = sorted({j["prompt_file"] for j in JOBS
                  if not (repo / j["prompt_file"]).exists()})
for name in missing:
    print(f"  MISSING PROMPT  {name}")
if missing:
    print()

TODO = []
for j in JOBS:
    done = ws.run_done(RUN_SET, j["run_id"])
    if not done:
        TODO.append(j)
    print(f"  {'done   ' if done else 'to run '} {j['run_id']}")

hours = sum(HOURS_PER[j["latent_frames"]] for j in TODO)
print(f"\n{len(TODO)} of {len(JOBS)} left   ~{hours:.1f} GPU-hours   "
      f"~{hours * 14:.0f} units")
print(f"\nstatus written to {ws.write_status()}")

## 4 · Generate

The only cell that costs GPU time. Each video takes 20–24 minutes, the manifest
is written even on a crash or interrupt, and `STATUS.md` is refreshed after each
one. Safe to stop and re-run at any point.

In [ ]:
if missing:
    raise SystemExit("Write the missing prompt files first (see cell 3).")
if not TODO:
    raise SystemExit("Nothing left to run.")

for n, j in enumerate(TODO, 1):
    run_id = j["run_id"]
    print(f"\n{'=' * 70}\n[{n}/{len(TODO)}] {run_id}\n{'=' * 70}")

    # Colab can reassign the runtime between runs; never mix hardware in one set.
    now = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    assert now == gpu_name, f"GPU changed mid-batch: {gpu_name} -> {now}"

    local_mp4 = Path(f"/content/out/{run_id}/video.mp4")
    local_mp4.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", "brainrot_inference.py",
               "--prompt_file", j["prompt_file"],
               "--checkpoint_path", str(CHECKPOINT),
               "--output_file", str(local_mp4),
               "--num_output_frames", str(j["latent_frames"]),
               "--sampling_steps", str(j["sampling_steps"]),
               "--seed", str(j["seed"]),
               "--mode", j["mode"],
               "--config_path", j["config_path"]]

    record = dict(
        j, run_set=RUN_SET, platform="colab",
        prompt_sha256=hashlib.sha256((repo / j["prompt_file"]).read_bytes()).hexdigest(),
        config=(repo / j["config_path"]).read_text(),
        command=command,
        gpu=gpu_name, gpu_total_gb=round(gpu_gb, 1),
        memory_path=MEMORY_PATH,           # resident vs dynamic_swap
        flash_attn=FLASH_ATTN,             # None means the SDPA fallback
        torch_version=torch.__version__,
        started_at=cl.utcnow(), status="running")
    cl.write_json(ws.manifest(RUN_SET, run_id), record)

    torch.cuda.reset_peak_memory_stats()
    began = time.monotonic()
    try:
        with open(ws.log(RUN_SET, run_id), "w", encoding="utf-8") as log:
            proc = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:        # streamed, so a crash leaves a trail
                log.write(line); log.flush(); print(line, end="")
            record["exit_code"] = proc.wait()

        if record["exit_code"] == 0 and local_mp4.exists():
            final = ws.video(RUN_SET, run_id)
            shutil.copy2(local_mp4, final)
            sidecar = local_mp4.with_suffix(".schedule.json")
            if sidecar.exists():
                shutil.copy2(sidecar, ws.schedule(RUN_SET, run_id))
            record.update(status="succeeded", output_path=str(final),
                          output_bytes=final.stat().st_size,
                          output_sha256=cl.sha256_file(final))
        else:
            record["status"] = "failed"
    except KeyboardInterrupt:
        record["status"] = "interrupted"
        raise
    except Exception as error:
        record.update(status="failed", error=repr(error))
        raise
    finally:
        record["wall_seconds"] = round(time.monotonic() - began, 1)
        record["peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        record["finished_at"] = cl.utcnow()
        cl.write_json(ws.manifest(RUN_SET, run_id), record)
        ws.write_status()                   # Drive shows progress immediately

    print(f"\n-> {record['status']}  {record['wall_seconds']:.0f}s  "
          f"peak {record['peak_vram_gb']} GB")

print(f"\ndone - {ws.root / 'STATUS.md'}")

## 5 · Check

Re-verifies every file and reprints the status table. Scoring, statistics and
figures live in `colab_evaluate.ipynb`.

In [ ]:
rows = ws.inventory(RUN_SET)
if not rows:
    print("nothing generated yet")
else:
    width = max(len(r["run_id"]) for r in rows)
    print(f"{'run':<{width}}  {'ok':<4} {'wall':>7} {'path':<14} attention")
    for r in rows:
        print(f"{r['run_id']:<{width}}  {'yes' if r['verified'] else 'NO ':<4} "
              f"{(r['wall_seconds'] or 0):>6.0f}s {str(r['memory_path']):<14} "
              f"{r['flash_attn'] or 'sdpa'}")

    paths = {r["memory_path"] for r in rows if r["verified"]}
    kernels = {r["flash_attn"] for r in rows if r["verified"]}
    if len(paths) > 1 or len(kernels) > 1:
        print(f"\nWARNING - mixed execution paths {paths} / {kernels}. Runs made "
              "on different hardware or attention kernels are not comparable.")

    bad = [r["run_id"] for r in rows if not r["verified"]]
    print(f"\n{len(rows) - len(bad)} of {len(rows)} verified"
          + (f"; re-run cell 4 for: {', '.join(bad)}" if bad else ""))

ws.write_status()
print(f"\nStatus table: {ws.root / 'STATUS.md'}")
print(f"Videos:       {ws.runs / RUN_SET / 'video'}")